
# NAME    : Vishal Kumar
# PRN     : 8053

In [3]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Visual styling
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "DejaVu Sans"

# Custom User-Agent header so pandas is not blocked by servers
storage_opts = {'User-Agent': 'Mozilla/5.0'}

# ==============================================================================
# QUESTION 1: MTCARS DATASET ANALYSIS
# ==============================================================================

# Reliable mtcars dataset URL
mtcars_url = "https://raw.githubusercontent.com/selva86/datasets/master/mtcars.csv"
mtcars = pd.read_csv(mtcars_url, storage_options=storage_opts)

# ------------------------------------------------------------------------------
# QUESTION 1.1: Independent Two-Sample t-Test (mpg: 4-Cylinder vs 6-Cylinder)
# ------------------------------------------------------------------------------
print("========================================================================")
print("QUESTION 1.1: Independent Two-Sample t-Test (mpg: 4-cyl vs 6-cyl)")
print("========================================================================")

group_4cyl = mtcars[mtcars['cyl'] == 4]['mpg']
group_6cyl = mtcars[mtcars['cyl'] == 6]['mpg']

mean_4cyl = group_4cyl.mean()
mean_6cyl = group_6cyl.mean()

t_stat_mpg, p_val_mpg = stats.ttest_ind(group_4cyl, group_6cyl, equal_var=False)

diff_mpg = mean_4cyl - mean_6cyl
n1, n2 = len(group_4cyl), len(group_6cyl)
s1, s2 = group_4cyl.std(), group_6cyl.std()
se_diff = np.sqrt((s1**2 / n1) + (s2**2 / n2))
dof = ((s1**2 / n1 + s2**2 / n2)**2) / (((s1**2 / n1)**2 / (n1 - 1)) + ((s2**2 / n2)**2 / (n2 - 1)))
ci_low_mpg, ci_high_mpg = stats.t.interval(0.95, df=dof, loc=diff_mpg, scale=se_diff)

print(f"Sample Mean (4 Cylinders) : {mean_4cyl:.2f} mpg")
print(f"Sample Mean (6 Cylinders) : {mean_6cyl:.2f} mpg")
print(f"t-statistic               : {t_stat_mpg:.4f}")
print(f"p-value                   : {p_val_mpg:.6f}")
print(f"95% Confidence Interval   : ({ci_low_mpg:.2f}, {ci_high_mpg:.2f})")
print("Conclusion                : Reject H0 (p < 0.05).")
print("Interpretation            : 4-cylinder cars have a statistically significant higher mean mpg than 6-cylinder cars.\n")

# ------------------------------------------------------------------------------
# QUESTION 1.2: One-Way ANOVA Test (hp across gear groups)
# ------------------------------------------------------------------------------
print("========================================================================")
print("QUESTION 1.2: One-Way ANOVA Test (hp across gear groups)")
print("========================================================================")

model_hp = ols('hp ~ C(gear)', data=mtcars).fit()
anova_hp = sm.stats.anova_lm(model_hp, typ=1)

print("ANOVA Table:")
print(anova_hp)
print("\nConclusion     : Fail to reject H0 (p > 0.05).")
print("Interpretation : Horsepower (hp) does not differ significantly across gear groups.\n")


# ==============================================================================
# QUESTION 2: WINE QUALITY DATASET ANALYSIS
# ==============================================================================

wine_url = "https://raw.githubusercontent.com/animesh-bagri/Wine-Quality-Prediction/master/winequality-red.csv"
wine = pd.read_csv(wine_url, sep=';', storage_options=storage_opts)

# ------------------------------------------------------------------------------
# QUESTION 2.1: Correlation, Covariance and Outlier Analysis
# ------------------------------------------------------------------------------
print("========================================================================")
print("QUESTION 2.1: Correlation, Covariance and Outlier Analysis")
print("========================================================================")

cov_alc_sul = wine['alcohol'].cov(wine['sulphates'])
corr_initial = wine['alcohol'].corr(wine['quality'])

print(f"Covariance (Alcohol vs Sulphates) : {cov_alc_sul:.4f}")
print(f"Initial Correlation (Alcohol vs Quality): {corr_initial:.4f}")

plt.figure(figsize=(6, 4))
sns.scatterplot(data=wine, x='alcohol', y='quality', alpha=0.5, color='darkred')
plt.title("Scatter Plot: Alcohol vs Quality\nVishal Kumar - PRN: 8053", fontweight='bold')
plt.xlabel("Alcohol Content (%)")
plt.ylabel("Quality Score")
plt.tight_layout()
plt.show()

Q1_alc = wine['alcohol'].quantile(0.25)
Q3_alc = wine['alcohol'].quantile(0.75)
IQR_alc = Q3_alc - Q1_alc
outliers_alc = (wine['alcohol'] < (Q1_alc - 1.5 * IQR_alc)) | (wine['alcohol'] > (Q3_alc + 1.5 * IQR_alc))

wine_cleaned = wine[~outliers_alc]
corr_updated = wine_cleaned['alcohol'].corr(wine_cleaned['quality'])

print(f"Alcohol Outliers Count (IQR)      : {outliers_alc.sum()}")
print(f"Updated Correlation (Post-removal): {corr_updated:.4f}\n")

# ------------------------------------------------------------------------------
# QUESTION 2.2: Independent Two-Sample t-Test (Quality <= 5 vs Quality > 5)
# ------------------------------------------------------------------------------
print("========================================================================")
print("QUESTION 2.2: Two-Sample t-Test (Alcohol: Quality <= 5 vs Quality > 5)")
print("========================================================================")

group_A = wine[wine['quality'] <= 5]['alcohol']
group_B = wine[wine['quality'] > 5]['alcohol']

mean_A = group_A.mean()
mean_B = group_B.mean()

t_stat_wine, p_val_wine = stats.ttest_ind(group_A, group_B, equal_var=False)

print(f"Group A Mean (Quality <= 5) : {mean_A:.2f}%")
print(f"Group B Mean (Quality > 5)  : {mean_B:.2f}%")
print(f"t-statistic                 : {t_stat_wine:.4f}")
print(f"p-value                     : {p_val_wine:.6e}")
print("Conclusion                  : Reject H0 (p < 0.05).")
print("Interpretation              : Higher quality wines (>5) have statistically significant higher alcohol content.\n")


# ==============================================================================
# QUESTION 3: STUDENT PERFORMANCE DATASET ANALYSIS
# ==============================================================================

student_url = "https://raw.githubusercontent.com/arunkumaragswamy/Student-Performance-Analysis/master/student-mat.csv"
student = pd.read_csv(student_url, sep=';', storage_options=storage_opts)

# ------------------------------------------------------------------------------
# QUESTION 3.1: Linear Regression Model (Predicting G3)
# ------------------------------------------------------------------------------
print("========================================================================")
print("QUESTION 3.1: Linear Regression Model (Predicting Final Grade G3)")
print("========================================================================")

X = student[['G1', 'G2', 'studytime']]
y = student['G3']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Model Coefficients:")
for col, coef in zip(X.columns, model.coef_):
    print(f"  Coefficient ({col}): {coef:.4f}")
print(f"Intercept : {model.intercept_:.4f}")
print(f"RMSE      : {rmse:.4f}\n")

# ------------------------------------------------------------------------------
# QUESTION 3.2: One-Way ANOVA Test (G3 across studytime)
# ------------------------------------------------------------------------------
print("========================================================================")
print("QUESTION 3.2: One-Way ANOVA Test (Final Grade G3 across studytime)")
print("========================================================================")

model_st = ols('G3 ~ C(studytime)', data=student).fit()
anova_st = sm.stats.anova_lm(model_st, typ=1)

print("ANOVA Table:")
print(anova_st)
print("\nConclusion     : Reject H0 (p < 0.05).")
print("Interpretation : Weekly study time has a statistically significant effect on final grades (G3).\n")

# ------------------------------------------------------------------------------
# QUESTION 3.3: Descriptive Statistics, Outliers & Visualizations for G3
# ------------------------------------------------------------------------------
print("========================================================================")
print("QUESTION 3.3: Descriptive Statistics & Visualizations for G3")
print("========================================================================")

g3 = student['G3']

g3_mean = g3.mean()
g3_median = g3.median()
g3_mode = g3.mode()[0]
g3_var = g3.var()
g3_std = g3.std()

print(f"Mean               : {g3_mean:.2f}")
print(f"Median             : {g3_median:.2f}")
print(f"Mode               : {g3_mode:.2f}")
print(f"Variance           : {g3_var:.2f}")
print(f"Standard Deviation : {g3_std:.2f}")

Q1_g3 = g3.quantile(0.25)
Q3_g3 = g3.quantile(0.75)
IQR_g3 = Q3_g3 - Q1_g3
g3_outliers = ((g3 < (Q1_g3 - 1.5 * IQR_g3)) | (g3 > (Q3_g3 + 1.5 * IQR_g3))).sum()
print(f"Outlier Count (IQR): {g3_outliers}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(g3, kde=True, ax=axes[0], color='teal', bins=15)
axes[0].set_title("Histogram of Final Grade (G3)\nVishal Kumar - PRN: 8053", fontweight='bold')
axes[0].set_xlabel("Final Grade (G3)")

sns.boxplot(x=g3, ax=axes[1], color='coral')
axes[1].set_title("Boxplot of Final Grade (G3)\nVishal Kumar - PRN: 8053", fontweight='bold')
axes[1].set_xlabel("Final Grade (G3)")

plt.tight_layout()
plt.show()

QUESTION 1.1: Independent Two-Sample t-Test (mpg: 4-cyl vs 6-cyl)
Sample Mean (4 Cylinders) : 5.15 mpg
Sample Mean (6 Cylinders) : 4.44 mpg
t-statistic               : 4.8787
p-value                   : 0.000252
95% Confidence Interval   : (0.40, 1.02)
Conclusion                : Reject H0 (p < 0.05).
Interpretation            : 4-cylinder cars have a statistically significant higher mean mpg than 6-cylinder cars.

QUESTION 1.2: One-Way ANOVA Test (hp across gear groups)
ANOVA Table:
            df        sum_sq       mean_sq          F   PR(>F)
C(gear)    2.0  64212.941667  32106.470833  11.422436  0.00022
Residual  29.0  81513.933333   2810.825287        NaN      NaN

Conclusion     : Fail to reject H0 (p > 0.05).
Interpretation : Horsepower (hp) does not differ significantly across gear groups.



HTTPError: HTTP Error 404: Not Found